In [61]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [62]:
%pip install numpy
%pip install scipy
%pip install matplotlib
%pip install pandas

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrad

In [63]:
import numpy
import pandas
from src.validate import Validate
from src.district_population import DistrictPopulation

years = list(range(2015, 2025))
Kampala = [1200, 1250, 1300, 1350, 1420, 1500, 1580, 1650, 1720, 1800]
Wakiso  = [ 950, 1000, 1070, 1150, 1220, 1300, 1390, 1480, 1570, 1670]
Gulu    = [ 320,  330,  345,  360,  375,  390,  410,  430,  455,  480]
Kabale    = [ 570,  590,  630,  640,  712,  733,  750,  765,  775,  790]
Masindi    = [270,  300,  310,  340,  360,  383,  390,  400,  410,  440]
populations = [Kampala, Wakiso, Gulu, Kabale, Masindi]
districts = ["Kampala", "Wakiso", 'Gulu', 'Kabale', 'Masindi']
models = ['Linear', 'Cagr','Fibonacci']

#create validate object
validate  = Validate()

all_populations = {'Kampala': Kampala, 'Wakiso': Wakiso, 'Gulu': Gulu, 'Kabale':Kabale, 'Masindi':Masindi}
#Print out numpy, statistics numbers by district
results = []

try:
    for district_name, populations in all_populations.items():
        district_stat_obj = DistrictPopulation(validate, numpy.array(years),numpy.array(populations),district_name)
        
        results.append({
            'District': district_name,
            'Module': '(Numpy)',
            'Mean': district_stat_obj.mean(),
            'Median': district_stat_obj.median(),
            'Variance': district_stat_obj.variance(),
            'Std Dev': district_stat_obj.standard_deviation()
            })

        results.append({
            'District': district_name,
            'Module': '(Statistics)',
            'Mean': district_stat_obj.mean('statistics'),
            'Median': district_stat_obj.median('statistics'),
            'Variance': district_stat_obj.variance('statistics'),
            'Std Dev': district_stat_obj.standard_deviation('statistics')
            })

    df = pandas.DataFrame(results).round(2)

    print(df.to_string(index=False))
except Exception as e:
    print("Error: ", e) 


District       Module   Mean  Median  Variance  Std Dev
 Kampala      (Numpy) 1477.0  1460.0  38341.00   195.81
 Kampala (Statistics) 1477.0  1460.0  42601.11   195.81
  Wakiso      (Numpy) 1280.0  1260.0  54060.00   232.51
  Wakiso (Statistics) 1280.0  1260.0  60066.67   232.51
    Gulu      (Numpy)  389.5   382.5   2597.25    50.96
    Gulu (Statistics)  389.5   382.5   2885.83    50.96
  Kabale      (Numpy)  695.5   722.5   5898.05    76.80
  Kabale (Statistics)  695.5   722.5   6553.39    76.80
 Masindi      (Numpy)  360.3   371.5   2652.81    51.51
 Masindi (Statistics)  360.3   371.5   2947.57    51.51


The statistics.variance and  np.var give different numbers because statistics.variance() calculates sample variance, while numpy.var() calculates population variance by default
- Numpy.var treats the data input as the complete set of values, so it divides the total squared differences from the mean by the number of observations, N. 
- Python's statistics.variance() treats the data as a sample taken from a larger population, so it divides by N − 1. Because N − 1 is smaller than N, the resulting variance is larger

The ddof argument in NumPy controls this difference: ddof=0 means divide by N, while ddof=1 means divide by N − 1. Therefore, statistics.variance() and NumPy's variance with ddof=1 produce the same result

### Growth Rate and CAGR section

In [64]:
cagr_by_district = {}

all_populations = {'Kampala': Kampala, 'Wakiso': Wakiso, 'Gulu': Gulu, 'Kabale':Kabale, 'Masindi':Masindi}
try:
        #print out growth rates and CAGR by district
        for district_name, populations in all_populations.items():
                district_growth_obj = DistrictPopulation(validate, numpy.array(years), numpy.array(populations), district_name)
                print(repr(district_growth_obj), ':Annual Growth Rate')
                growth = district_growth_obj.year_on_year_growth()
                growth_table = pandas.DataFrame([growth])
                print(growth_table.to_string(index=False))
                cagr_by_district[district_name] = district_growth_obj.compound_annual_growth_rate()
                print('compound annual growth rate(CAGR):' + str(cagr_by_district[district_name]), '\n')
        
    
        #Get the fastest growing district
        fastest_growing_district = max(cagr_by_district, key=cagr_by_district.get)
        print("\n" , f"NOTE: The fastest growing distict is: {fastest_growing_district}") 
except Exception as e:
    print("Error: ", e) 

Kampala district :Annual Growth Rate
2015   2016 2017   2018   2019   2020   2021  2022   2023   2024
0.0% 4.167% 4.0% 3.846% 5.185% 5.634% 5.333% 4.43% 4.242% 4.651%
Error:  Can't instantiate abstract class Cagr with abstract method fit


### Prediction Models section

In [71]:
#prediction year
prediction_years = years[7:]

training_populations = {'Kampala': Kampala[:7], 'Wakiso': Wakiso[:7], 'Gulu': Gulu[:7], 'Kabale':Kabale[:7], 'Masindi':Masindi[:7]}

#target data
actual_populations = {'Kampala': Kampala[7:], 'Wakiso': Wakiso[7:], 'Gulu': Gulu[7:], 'Kabale':Kabale[7:], 'Masindi':Masindi[7:]}
try:
    predictions = {}
    for district_name, populations in all_populations.items():
        training_data = training_populations[district_name]
        actual_data = actual_populations[district_name]
        district_model_obj = DistrictPopulation(validate, numpy.array(years[:7]), numpy.array(training_data), district_name)
        #new inner dictionary for the district
        predictions[district_name] = {}
        #include actual data
        predictions[district_name]['Actual-data'] = {}
        predictions[district_name]['Actual-data']['forecast'] = actual_data
        predictions[district_name]['Actual-data']['metrics'] = {'MAE':'N/A', 'RMSE': 'N/A', 'MAPE': 'N/A'}
        
        #handle the predictions and mae, rsme, mape
        for model in models:
            predictions[district_name][model] =  {}
            forecast = district_model_obj.prediction(len(actual_data), model)
            predictions[district_name][model]['forecast'] = forecast
            predictions[district_name][model]['metrics'] =  district_model_obj.prediction_metrics(numpy.array(actual_data),forecast)
        
        predictions[district_name]['best-model'] = district_model_obj.best_model(predictions[district_name])      

    print("Model Training Results")

    for district, inner_models in predictions.items():

        results = []
        for i_model, m_data in inner_models.items():
            #skip this, its does not have details
            if i_model == 'best-model':
                continue

            row = {'Model': i_model,**dict(zip(prediction_years, m_data['forecast']))}

            if 'metrics' in m_data:
                metrics = m_data['metrics']
                row.update({'MAE': metrics['MAE'], 'RMSE': metrics['RMSE'],'MAPE': metrics['MAPE']})

            results.append(row)

        df = pandas.DataFrame(results).round(2)

        print(f"\n{district}")
        print(df.to_string(index=False))
        print(f"Best Model: {inner_models['best-model']}")
except Exception as e:
    print("Error: ", e) 

Model Training Results

Kampala
      Model    2022    2023    2024     MAE     RMSE   MAPE
Actual-data 1650.00 1720.00 1800.00     N/A      N/A    N/A
     Linear 1622.86 1685.71 1748.57   37.62    38.97   2.17
       Cagr 1654.13 1731.74 1812.99    9.62    10.39   0.55
  Fibonacci 3160.00 2370.00 2633.33  997.78  1064.12  58.53
Best Model: Cagr

Wakiso
      Model    2022    2023    2024     MAE    RMSE   MAPE
Actual-data 1480.00 1570.00 1670.00     N/A     N/A    N/A
     Linear 1450.00 1523.93 1597.86    49.4   52.37   3.09
       Cagr 1481.03 1578.02 1681.36     6.8    8.05   0.42
  Fibonacci 2780.00 2085.00 2316.67  820.56  889.46  53.12
Best Model: Cagr

Gulu
      Model   2022   2023   2024     MAE    RMSE   MAPE
Actual-data 430.00 455.00 480.00     N/A     N/A    N/A
     Linear 421.43 436.43 451.43   18.57   20.29   4.01
       Cagr 427.29 445.31 464.09    9.44   10.87   2.02
  Fibonacci 820.00 615.00 683.33  251.11  270.21  56.07
Best Model: Cagr

Kabale
      Model    2022 

### Best model forecast


In [72]:
#use best models to do prediction
# print(predictions)
bm_prediction_years = [2025, 2026, 2027, 2028, 2029]
bm_prediction = {}
print(predictions)
for d in districts:
    best_model = predictions[d]['best-model']
    districtObj = DistrictPopulation(validate, numpy.array(years), numpy.array(all_populations[d]), d)
    bm_prediction[d] = districtObj.prediction(len(bm_prediction_years), best_model)

#one row per district, one column per year
df = pandas.DataFrame.from_dict(bm_prediction, orient='index', columns=bm_prediction_years).round(2)
df.insert(0, 'Best Model', [predictions[d]['best-model'] for d in df.index])
df.index.name = 'District'

print(df.to_string())

{'Kampala': {'Actual-data': {'forecast': [1650, 1720, 1800], 'metrics': {'MAE': 'N/A', 'RMSE': 'N/A', 'MAPE': 'N/A'}}, 'Linear': {'forecast': array([1622.85714286, 1685.71428571, 1748.57142857]), 'metrics': {'MAE': np.float64(37.62), 'RMSE': np.float64(38.97), 'MAPE': np.float64(2.17)}}, 'Cagr': {'forecast': array([1654.13033837, 1731.73871919, 1812.98832502]), 'metrics': {'MAE': np.float64(9.62), 'RMSE': np.float64(10.39), 'MAPE': np.float64(0.55)}}, 'Fibonacci': {'forecast': array([3160.        , 2370.        , 2633.33333333]), 'metrics': {'MAE': np.float64(997.78), 'RMSE': np.float64(1064.12), 'MAPE': np.float64(58.53)}}, 'best-model': 'Cagr'}, 'Wakiso': {'Actual-data': {'forecast': [1480, 1570, 1670], 'metrics': {'MAE': 'N/A', 'RMSE': 'N/A', 'MAPE': 'N/A'}}, 'Linear': {'forecast': array([1450.        , 1523.92857143, 1597.85714286]), 'metrics': {'MAE': np.float64(49.4), 'RMSE': np.float64(52.37), 'MAPE': np.float64(3.09)}}, 'Cagr': {'forecast': array([1481.02821645, 1578.01768195, 

### compare the variance of the actual series against the variance of the forecast series

In [73]:
rows = []
for district_name, populations in all_populations.items():
    actual = numpy.array(populations)
    forecast = bm_prediction[district_name]

    #level variance: full actual series (10 years), last 5 actual years, and the 5 forecast years
    actual_var = DistrictPopulation(validate, numpy.array(years), actual, district_name).variance()
    recent_var = DistrictPopulation(validate, numpy.array(years[-5:]), actual[-5:], district_name).variance()
    forecast_var = DistrictPopulation(validate, numpy.array(bm_prediction_years), forecast, district_name).variance()

    #year-to-year changes remove the trend: 2016-2024 for actual, 2025-2029 for forecast (starting from 2024)
    actual_changes = numpy.diff(actual)
    forecast_changes = numpy.diff(numpy.append(actual[-1], forecast))
    actual_change_var = DistrictPopulation(validate, numpy.array(years[1:]), actual_changes, district_name).variance()
    forecast_change_var = DistrictPopulation(validate, numpy.array(bm_prediction_years), forecast_changes, district_name).variance()

    rows.append({
        'District': district_name,
        'Best Model': predictions[district_name]['best-model'],
        'Var actual 2015-24': actual_var,
        'Var actual 2020-24': recent_var,
        'Var forecast 2025-29': forecast_var,
        'Ratio (5y vs 5y)': forecast_var / recent_var,
        'Var yearly change actual': actual_change_var,
        'Var yearly change forecast': forecast_change_var,
    })

print(pandas.DataFrame(rows).round(2).to_string(index=False))

District Best Model  Var actual 2015-24  Var actual 2020-24  Var forecast 2025-29  Ratio (5y vs 5y)  Var yearly change actual  Var yearly change forecast
 Kampala       Cagr            38341.00            10960.00              17286.15              1.58                    155.56                       33.55
  Wakiso       Cagr            54060.00            16936.00              32105.09              1.90                    200.00                      118.51
    Gulu       Cagr             2597.25             1016.00               1229.24              1.21                     22.84                        2.39
  Kabale       Cagr             5898.05              389.04               2044.74              5.26                    354.02                        2.59
 Masindi     Linear             2652.81              396.64                627.65              1.58                     85.21                        0.00


### Variance of the actual values compared with the forecasts (2022-2024)

Variance shows how much the forecasted values change compared with the actual values from 2022–2024.

A ratio close to 1 means the forecast has a similar level of variation to the actual data.
A ratio below 1 means the forecast varies less than the actual data.
A ratio above 1 means the forecast varies more than the actual data.

In this data, Linear and CAGR generally have variances closer to the actual values, while Fibonacci has much higher variance, meaning its forecasts fluctuate much more than the actual population values.